# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\u00b2 dataset using the `mlcroissant` library and references all elements by their schema `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed (run in notebook cell)
!pip install mlcroissant

## 1. Data Loading
Load the schema, metadata, and records using `mlcroissant`. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema source URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Dataset object
dataset = mlc.Dataset(croissant_url)

# Print key metadata (not using subscript notation, only attributes)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Croissant Schema URL: {croissant_url}")

## 2. Data Overview
List record sets, their `@id`s, and fields (with `@id`) for exploration.

In [ ]:
# List all RecordSets with their @id and corresponding schema (field) @id's
print('Record sets in this dataset:')
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    print("  Fields/Columns:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

# Save list of record_set @ids for later use
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
if not record_set_ids:
    print("No record sets found in this dataset; check 'dataset.metadata.record_sets' via Croissant schema for availability.")

## 3. Data Extraction
Load data from one or multiple record sets into DataFrames for further analysis. 
All references use record set and field `@id`s.

In [ ]:
dataframes = {}
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# For demonstration, try to read all discovered record sets
for rs_id in record_set_ids:
    print(f"Loading records from: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"...Loaded {len(df)} records with columns: {df.columns.tolist()}")
    except Exception as e:
        print(f"...Could not load {rs_id}: {e}")
    print()

# Show example from the first (main) record set, if available
if record_set_ids:
    first_id = record_set_ids[0]
    if first_id in dataframes:
        print(f"Columns for record set {first_id}:\n", dataframes[first_id].columns.tolist())
        dataframes[first_id].head()
    else:
        print(f"No DataFrame available for record set {first_id}")
else:
    print("There are no record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA steps: subset, normalize, and group by fields. All schema elements are referenced by `@id`.

In [ ]:
# Choose one record set for EDA (the first, if available)
if record_set_ids:
    record_set_id = record_set_ids[0]  # Use the main record set
    df = dataframes.get(record_set_id)

    if df is not None and not df.empty:
        print(f"EDA for record set @id: {record_set_id}")

        # Attempt to autodetect a numeric field (@id):
        numeric_field_id = None
        # Get fields from schema to match pd.DataFrame columns to field @ids
        rs_obj = [r for r in dataset.metadata.record_sets if r.id == record_set_id][0]
        for field in rs_obj.fields:
            # Pick a column that is likely numeric
            if field.data_type in ('Float', 'Number', 'Integer') and field.id in df.columns:
                numeric_field_id = field.id
                break

        if numeric_field_id is None:
            print("No numeric field found for EDA. Available columns are:", df.columns.tolist())
        else:
            print(f"Using numeric field: {numeric_field_id}")

            # Convert to numeric just in case
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() is not None else 1
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean)")

            col_norm = f"{numeric_field_id}_normalized"
            filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(filtered_df[[numeric_field_id, col_norm]].head())

            # Try grouping by a categorical field
            group_field_id = None
            for field in rs_obj.fields:
                # Choose first non-numeric field
                if field.data_type not in ('Float', 'Number', 'Integer') and field.id in df.columns:
                    group_field_id = field.id
                    break
            if group_field_id is not None:
                print(f"Grouping by: {group_field_id}")
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(grouped_df.head())
            else:
                print("No suitable grouping field available.")
    else:
        print("No data available for EDA in the DataFrame.")
else:
    print("No record set detected for EDA.")

## 5. Visualization
Visualize distributions of key numeric fields and aggregates in the chosen record set using matplotlib where possible.

In [ ]:
# Histogram and boxplot for the normalized field
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes.get(record_set_id)
    if df is not None:
        rs_obj = [r for r in dataset.metadata.record_sets if r.id == record_set_id][0]

        # Try to get the same numeric_field_id as before
        numeric_field_id = None
        for field in rs_obj.fields:
            if field.data_type in ('Float', 'Number', 'Integer') and field.id in df.columns:
                numeric_field_id = field.id
                break
        if numeric_field_id is not None:
            # Convert to numeric for plotting
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            plt.figure(figsize=(8, 4))
            sns.histplot(df[numeric_field_id].dropna(), kde=True)
            plt.title(f"Distribution of {numeric_field_id}")
            plt.xlabel(numeric_field_id)
            plt.show()

            # Boxplot by group_field (if available)
            group_field_id = None
            for field in rs_obj.fields:
                # Pick a non-numeric field for grouping
                if field.data_type not in ('Float', 'Number', 'Integer') and field.id in df.columns:
                    group_field_id = field.id
                    break
            if group_field_id is not None:
                plt.figure(figsize=(8,4))
                sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
                plt.title(f"{numeric_field_id} by {group_field_id}")
                plt.xticks(rotation=90)
                plt.tight_layout()
                plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
This notebook demonstrated structured exploration of the FAIR\u00b2 dataset by referencing record sets and fields strictly with their `@id` via the `mlcroissant` library. 

Typical data science workflows such as extracting key statistics, subsetting and normalizing data, and generating group-based summaries and basic visualizations can be performed directly from Croissant-remediated data tables. 
For further analysis (clustering, modeling, etc.), users are now equipped with DataFrames whose columns map directly to fields' `@id`s for unambiguous, schema-driven workflows.